# Administrative Boundaries Validation

## Project
Machine Learning-Based Early Warning System for Poultry Disease Outbreaks in Nigeria

## Objective

This notebook validates the geographic relationship between the WAHIS
analytical table and the Nigeria ADM1 administrative boundary dataset.

The validation focuses on:

1. Inspecting the WAHIS administrative division names.
2. Inspecting the ADM1 GeoJSON boundary names.
3. Standardising names for comparison.
4. Identifying exact geographic matches.
5. Identifying unmatched WAHIS administrative divisions.
6. Identifying boundary divisions without WAHIS records.
7. Assessing geographic coverage.
8. Creating a validated geographic lookup table for later data integration.

## Important methodological principle

Geographic names are not changed arbitrarily.

The original WAHIS administrative division values are preserved, while
standardised names are created separately for matching purposes.

Automatic fuzzy matching is used only to identify possible candidates.
It is not treated as confirmed geographic matching without validation.

The final geographic integration will be performed later in the
Data Integration notebook.

In [2]:
# ============================================================
# NOTEBOOK 03 — ADMINISTRATIVE BOUNDARIES VALIDATION
# SECTION 1 — LOAD DATASETS
# ============================================================

from pathlib import Path
import json
import re
import unicodedata

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Locate the project root.
#
# The notebook may be opened from different working
# directories, so we search upward for the project structure.
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (
        (PROJECT_ROOT / "data").exists()
        and (PROJECT_ROOT / "src").exists()
    ):
        break

    PROJECT_ROOT = PROJECT_ROOT.parent


print("Project root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# 2. Define the WAHIS analytical-table path.
# ------------------------------------------------------------

WAHIS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "wahis_analytical_table.csv"
)


# ------------------------------------------------------------
# 3. Locate the administrative boundary GeoJSON.
#
# The boundary file may be stored in the project root,
# data directory, geo/ folder, or another subdirectory.
#
# We therefore search recursively rather than assuming
# a fixed location.
# ------------------------------------------------------------

BOUNDARY_FILENAME = "geoBoundaries-NGA-ADM1.geojson"

boundary_candidates = [
    PROJECT_ROOT / BOUNDARY_FILENAME,
    PROJECT_ROOT / "data" / BOUNDARY_FILENAME,
    PROJECT_ROOT / "data" / "raw" / BOUNDARY_FILENAME,
    PROJECT_ROOT / "data" / "external" / BOUNDARY_FILENAME,
    PROJECT_ROOT / "geo" / BOUNDARY_FILENAME,
    PROJECT_ROOT / "geodata" / BOUNDARY_FILENAME,
]


# Check the most likely locations first.
BOUNDARY_PATH = None

for candidate in boundary_candidates:
    if candidate.exists():
        BOUNDARY_PATH = candidate
        break


# If it was not found in the expected locations,
# search the entire project directory.
if BOUNDARY_PATH is None:

    matches = list(
        PROJECT_ROOT.rglob(BOUNDARY_FILENAME)
    )

    if len(matches) > 0:
        BOUNDARY_PATH = matches[0]


# ------------------------------------------------------------
# 4. Confirm that the required files exist.
# ------------------------------------------------------------

assert WAHIS_PATH.exists(), (
    f"WAHIS analytical table not found:\n{WAHIS_PATH}"
)

assert BOUNDARY_PATH is not None, (
    "Administrative boundary file not found.\n\n"
    f"Expected filename:\n{BOUNDARY_FILENAME}\n\n"
    f"Project root searched:\n{PROJECT_ROOT}\n\n"
    "Please place the GeoJSON file somewhere inside the "
    "project directory and run this cell again."
)


print("\nAdministrative boundary file found:")
print(BOUNDARY_PATH)


# ------------------------------------------------------------
# 5. Load the WAHIS analytical table.
# ------------------------------------------------------------

df_wahis = pd.read_csv(
    WAHIS_PATH
)


# ------------------------------------------------------------
# 6. Load the GeoJSON boundary file.
#
# We use Python's standard json library here rather than
# requiring GeoPandas just to inspect and validate the
# administrative names.
# ------------------------------------------------------------

with open(
    BOUNDARY_PATH,
    "r",
    encoding="utf-8"
) as f:
    geojson = json.load(f)


# ------------------------------------------------------------
# 7. Basic inspection.
# ------------------------------------------------------------

print("=" * 60)
print("WAHIS ANALYTICAL TABLE")
print("=" * 60)

print(f"Rows: {len(df_wahis):,}")
print(f"Columns: {len(df_wahis.columns)}")


print("\nAdministrative divisions in WAHIS:")

print(
    df_wahis["administrative_division"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)


print("\n" + "=" * 60)
print("GEOJSON ADMINISTRATIVE BOUNDARIES")
print("=" * 60)

print(
    "GeoJSON type:",
    geojson.get("type")
)

print(
    "Number of features:",
    len(geojson.get("features", []))
)


# ------------------------------------------------------------
# 8. Inspect the properties of the first boundary feature.
#
# This allows us to determine which property contains the
# administrative division name instead of assuming a property
# name.
# ------------------------------------------------------------

features = geojson.get("features", [])

assert len(features) > 0, (
    "GeoJSON contains no features."
)

first_properties = features[0].get(
    "properties",
    {}
)

print(
    "\nProperties available in the first GeoJSON feature:"
)

print(
    list(first_properties.keys())
)

print(
    "\nFirst feature properties:"
)

print(first_properties)

Project root:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria

Administrative boundary file found:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\raw\geoBoundaries-NGA-ADM1.geojson
WAHIS ANALYTICAL TABLE
Rows: 1,180
Columns: 22

Administrative divisions in WAHIS:
['Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa', 'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti', 'Enugu', 'Federal Capital Territory', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Nassarawa', 'Niger', 'Nigeria', 'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba', 'Yobe', 'Zamfara']

GEOJSON ADMINISTRATIVE BOUNDARIES
GeoJSON type: FeatureCollection
Number of features: 37

Properties available in the first GeoJSON feature:
['shapeName', 'shapeISO', 'shapeID', 'shapeGroup', 'shapeType']

First feature properties:
{'shapeName': 'Cross River', 'shapeISO': 'NG-CR', 'shapeID': '27

In [3]:
# ============================================================
# NOTEBOOK 03 — ADMINISTRATIVE BOUNDARIES VALIDATION
# SECTION 2 — EXTRACT BOUNDARY NAMES
# ============================================================

# ------------------------------------------------------------
# 1. Examine all property names across the GeoJSON features.
#
# Different GeoJSON datasets can contain slightly different
# schemas, so we collect the available property keys.
# ------------------------------------------------------------

property_keys = set()

for feature in features:
    properties = feature.get("properties", {})
    property_keys.update(properties.keys())

print("All GeoJSON property names:")
print(sorted(property_keys))


# ------------------------------------------------------------
# 2. Identify likely administrative-name fields.
#
# We look for property names containing terms commonly used
# for administrative names.
# ------------------------------------------------------------

name_candidates = [
    key
    for key in property_keys
    if any(
        term in key.lower()
        for term in [
            "name",
            "shape",
            "admin",
            "region",
            "province",
            "state"
        ]
    )
]

print("\nPotential administrative name fields:")
print(name_candidates)


# ------------------------------------------------------------
# 3. Automatically select a likely name field if possible.
#
# We prioritise common geoBoundaries naming conventions.
# If none is found, the cell stops so that we can inspect
# the schema rather than using the wrong field.
# ------------------------------------------------------------

preferred_name_fields = [
    "shapeName",
    "shapeName_1",
    "NAME_1",
    "name",
    "NAME",
    "admin1Name",
    "ADM1_NAME"
]

boundary_name_field = None

for candidate in preferred_name_fields:
    if candidate in property_keys:
        boundary_name_field = candidate
        break


if boundary_name_field is None:
    raise ValueError(
        "Could not automatically identify the ADM1 name field. "
        "Inspect the property names printed above and select "
        "the appropriate administrative-name field."
    )


print(
    f"\nSelected ADM1 name field: {boundary_name_field}"
)


# ------------------------------------------------------------
# 4. Extract the administrative division names.
# ------------------------------------------------------------

boundary_records = []

for feature in features:
    properties = feature.get("properties", {})

    boundary_name = properties.get(
        boundary_name_field
    )

    boundary_records.append({
        "boundary_name": boundary_name
    })


df_boundaries = pd.DataFrame(boundary_records)


# ------------------------------------------------------------
# 5. Remove empty names and duplicate names.
# ------------------------------------------------------------

df_boundaries = (
    df_boundaries[
        df_boundaries["boundary_name"].notna()
    ]
    .drop_duplicates()
    .sort_values("boundary_name")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Display the boundary names.
# ------------------------------------------------------------

print(
    f"\nUnique ADM1 boundary divisions: "
    f"{len(df_boundaries):,}"
)

display(df_boundaries)


# ------------------------------------------------------------
# 7. Validate that the expected ADM1 boundary structure
#    exists.
# ------------------------------------------------------------

assert len(df_boundaries) > 0, (
    "No ADM1 boundary names were extracted."
)

print(
    "\nADM1 boundary-name extraction completed successfully."
)

All GeoJSON property names:
['shapeGroup', 'shapeID', 'shapeISO', 'shapeName', 'shapeType']

Potential administrative name fields:
['shapeISO', 'shapeType', 'shapeName', 'shapeID', 'shapeGroup']

Selected ADM1 name field: shapeName

Unique ADM1 boundary divisions: 37


,boundary_name
0,Abia
1,Abuja Federal Capital Territory
2,Adamawa
3,Akwa Ibom
4,Anambra
5,Bauchi
6,Bayelsa
7,Benue
8,Borno
9,Cross River



ADM1 boundary-name extraction completed successfully.


In [4]:
# ============================================================
# NOTEBOOK 03 — ADMINISTRATIVE BOUNDARIES VALIDATION
# SECTION 3 — STANDARDISE AND MATCH ADMINISTRATIVE NAMES
# ============================================================

# ------------------------------------------------------------
# 1. Create a reusable function for standardising geographic
#    names.
#
# The original names are NOT overwritten.
# This function creates comparison keys only.
# ------------------------------------------------------------

def standardise_geographic_name(value):
    """
    Create a conservative standardised geographic name.

    The function:
    - converts values to strings;
    - removes accents;
    - converts to lowercase;
    - replaces punctuation with spaces;
    - normalises whitespace.

    It does NOT replace geographic names with guessed
    alternatives.
    """

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Remove accents/diacritics.
    value = unicodedata.normalize(
        "NFKD",
        value
    ).encode(
        "ascii",
        "ignore"
    ).decode("ascii")

    # Convert to lowercase.
    value = value.lower()

    # Replace punctuation and separators with spaces.
    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )

    # Collapse repeated whitespace.
    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


# ------------------------------------------------------------
# 2. Create standardised matching keys.
# ------------------------------------------------------------

df_wahis["admin_division_standard"] = (
    df_wahis["administrative_division"]
    .apply(standardise_geographic_name)
)

df_boundaries["boundary_name_standard"] = (
    df_boundaries["boundary_name"]
    .apply(standardise_geographic_name)
)


# ------------------------------------------------------------
# 3. Obtain unique WAHIS administrative divisions.
# ------------------------------------------------------------

wahis_admin = (
    df_wahis[
        [
            "administrative_division",
            "admin_division_standard"
        ]
    ]
    .dropna(subset=["administrative_division"])
    .drop_duplicates()
    .sort_values("administrative_division")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Create the set of boundary names.
# ------------------------------------------------------------

boundary_names = set(
    df_boundaries["boundary_name_standard"]
    .dropna()
)


# ------------------------------------------------------------
# 5. Identify exact standardised matches.
# ------------------------------------------------------------

wahis_admin["exact_boundary_match"] = (
    wahis_admin["admin_division_standard"]
    .isin(boundary_names)
)


# ------------------------------------------------------------
# 6. Calculate geographic matching statistics.
# ------------------------------------------------------------

total_wahis_admin = len(wahis_admin)

matched_admin = (
    wahis_admin["exact_boundary_match"]
    .sum()
)

unmatched_admin = (
    total_wahis_admin
    - matched_admin
)

match_rate = (
    matched_admin
    / total_wahis_admin
    * 100
    if total_wahis_admin > 0
    else 0
)


# ------------------------------------------------------------
# 7. Display the matching summary.
# ------------------------------------------------------------

print("=" * 60)
print("ADMINISTRATIVE NAME MATCHING")
print("=" * 60)

print(
    f"Unique WAHIS administrative divisions: "
    f"{total_wahis_admin:,}"
)

print(
    f"Exact standardised matches: "
    f"{matched_admin:,}"
)

print(
    f"Unmatched WAHIS divisions: "
    f"{unmatched_admin:,}"
)

print(
    f"Geographic matching rate: "
    f"{match_rate:.2f}%"
)


# ------------------------------------------------------------
# 8. Display matched administrative divisions.
# ------------------------------------------------------------

print("\nMatched administrative divisions:")

display(
    wahis_admin[
        wahis_admin["exact_boundary_match"]
    ]
)


# ------------------------------------------------------------
# 9. Display unmatched administrative divisions.
#
# These require investigation before geographic integration.
# ------------------------------------------------------------

print("\nUnmatched WAHIS administrative divisions:")

display(
    wahis_admin[
        ~wahis_admin["exact_boundary_match"]
    ]
)

ADMINISTRATIVE NAME MATCHING
Unique WAHIS administrative divisions: 39
Exact standardised matches: 36
Unmatched WAHIS divisions: 3
Geographic matching rate: 92.31%

Matched administrative divisions:


,administrative_division,admin_division_standard,exact_boundary_match
0,Abia,abia,True
1,Adamawa,adamawa,True
2,Akwa Ibom,akwa ibom,True
3,Anambra,anambra,True
4,Bauchi,bauchi,True
5,Bayelsa,bayelsa,True
6,Benue,benue,True
7,Borno,borno,True
8,Cross River,cross river,True
9,Delta,delta,True



Unmatched WAHIS administrative divisions:


,administrative_division,admin_division_standard,exact_boundary_match
14,Federal Capital Territory,federal capital territory,False
26,Nassarawa,nassarawa,False
28,Nigeria,nigeria,False


In [5]:
# ============================================================
# NOTEBOOK 03 — ADMINISTRATIVE BOUNDARIES VALIDATION
# SECTION 4 — INVESTIGATE UNMATCHED NAMES
# ============================================================

from difflib import SequenceMatcher


# ------------------------------------------------------------
# 1. Create a similarity function.
# ------------------------------------------------------------

def similarity_score(name_a, name_b):
    """
    Return a similarity score between 0 and 1.

    A score of 1 means the strings are identical.
    Lower values indicate greater differences.
    """

    return SequenceMatcher(
        None,
        str(name_a),
        str(name_b)
    ).ratio()


# ------------------------------------------------------------
# 2. Generate candidate boundary names for unmatched
#    WAHIS administrative divisions.
#
# IMPORTANT:
# These are suggestions only.
# They are NOT automatically accepted as geographic matches.
# ------------------------------------------------------------

unmatched = wahis_admin[
    ~wahis_admin["exact_boundary_match"]
].copy()

boundary_name_list = (
    df_boundaries["boundary_name"]
    .dropna()
    .tolist()
)

candidate_rows = []

for _, row in unmatched.iterrows():

    wahis_name = row["administrative_division"]

    scores = [
        (
            boundary_name,
            similarity_score(
                wahis_name,
                boundary_name
            )
        )
        for boundary_name in boundary_name_list
    ]

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_candidates = scores[:3]

    candidate_rows.append({
        "wahis_name": wahis_name,
        "candidate_1": top_candidates[0][0]
            if len(top_candidates) > 0 else None,
        "score_1": top_candidates[0][1]
            if len(top_candidates) > 0 else None,
        "candidate_2": top_candidates[1][0]
            if len(top_candidates) > 1 else None,
        "score_2": top_candidates[1][1]
            if len(top_candidates) > 1 else None,
        "candidate_3": top_candidates[2][0]
            if len(top_candidates) > 2 else None,
        "score_3": top_candidates[2][1]
            if len(top_candidates) > 2 else None
    })


df_unmatched_candidates = pd.DataFrame(
    candidate_rows
)


# ------------------------------------------------------------
# 3. Display candidate matches.
# ------------------------------------------------------------

print(
    f"Unmatched WAHIS administrative divisions: "
    f"{len(df_unmatched_candidates)}"
)

if len(df_unmatched_candidates) > 0:

    display(
        df_unmatched_candidates
        .sort_values("score_1", ascending=False)
    )

else:

    print(
        "No unmatched administrative divisions "
        "require candidate investigation."
    )

Unmatched WAHIS administrative divisions: 3


,wahis_name,candidate_1,score_1,candidate_2,score_2,candidate_3,score_3
1,Nassarawa,Nasarawa,0.941176,Taraba,0.533333,Adamawa,0.500000
0,Federal Capital Territory,Abuja Federal Capital Territory,0.892857,Delta,0.266667,Adamawa,0.250000
2,Nigeria,Niger,0.833333,Jigawa,0.461538,Rivers,0.461538


In [6]:
# ============================================================
# NOTEBOOK 03 — ADMINISTRATIVE BOUNDARIES VALIDATION
# SECTION 5 — CREATE GEOGRAPHIC LOOKUP
# ============================================================

# ------------------------------------------------------------
# 1. Create a lookup table connecting each WAHIS administrative
#    division with its standardised geographic name.
# ------------------------------------------------------------

geographic_lookup = (
    wahis_admin[
        [
            "administrative_division",
            "admin_division_standard",
            "exact_boundary_match"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Add the corresponding boundary name where an exact
#    standardised match exists.
# ------------------------------------------------------------

boundary_mapping = (
    df_boundaries[
        [
            "boundary_name",
            "boundary_name_standard"
        ]
    ]
    .drop_duplicates()
    .set_index("boundary_name_standard")
    ["boundary_name"]
    .to_dict()
)


geographic_lookup["matched_boundary_name"] = (
    geographic_lookup[
        "admin_division_standard"
    ]
    .map(boundary_mapping)
)


# ------------------------------------------------------------
# 3. Add a clear validation status.
# ------------------------------------------------------------

geographic_lookup["geographic_match_status"] = np.where(
    geographic_lookup["exact_boundary_match"],
    "Matched",
    "Unmatched"
)


# ------------------------------------------------------------
# 4. Reorder columns for readability.
# ------------------------------------------------------------

geographic_lookup = geographic_lookup[
    [
        "administrative_division",
        "admin_division_standard",
        "matched_boundary_name",
        "geographic_match_status"
    ]
]


# ------------------------------------------------------------
# 5. Display the lookup.
# ------------------------------------------------------------

display(geographic_lookup)


# ------------------------------------------------------------
# 6. Save the lookup table.
#
# This lookup will be useful during Notebook 05,
# Data Integration.
# ------------------------------------------------------------

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "wahis_administrative_boundary_lookup.csv"
)

geographic_lookup.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"\nGeographic lookup saved to:\n{OUTPUT_PATH}"
)

,administrative_division,admin_division_standard,matched_boundary_name,geographic_match_status
0,Abia,abia,Abia,Matched
1,Adamawa,adamawa,Adamawa,Matched
2,Akwa Ibom,akwa ibom,Akwa Ibom,Matched
3,Anambra,anambra,Anambra,Matched
4,Bauchi,bauchi,Bauchi,Matched
5,Bayelsa,bayelsa,Bayelsa,Matched
6,Benue,benue,Benue,Matched
7,Borno,borno,Borno,Matched
8,Cross River,cross river,Cross River,Matched
9,Delta,delta,Delta,Matched



Geographic lookup saved to:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_administrative_boundary_lookup.csv


In [8]:
# ============================================================
# NOTEBOOK 03 — ADMINISTRATIVE BOUNDARIES VALIDATION
# SECTION 5 — FINALISE GEOGRAPHIC MAPPING
# ============================================================

# ------------------------------------------------------------
# IMPORTANT:
#
# The previous similarity analysis identified three unmatched
# WAHIS values:
#
# 1. Nassarawa
#    -> spelling variation of Nasarawa
#
# 2. Federal Capital Territory
#    -> GeoJSON uses Abuja Federal Capital Territory
#
# 3. Nigeria
#    -> country-level value, NOT an ADM1 administrative division
#
# We therefore resolve the first two explicitly and classify
# the country-level Nigeria record separately.
#
# We do NOT use the fuzzy similarity score to automatically
# assign geographic boundaries.
# ------------------------------------------------------------


# ------------------------------------------------------------
# 1. Create an explicit geographic-name mapping.
#
# Only known naming variations are included here.
# ------------------------------------------------------------

manual_geographic_mapping = {
    "Nassarawa": "Nasarawa",
    "Federal Capital Territory": "Abuja Federal Capital Territory"
}


# ------------------------------------------------------------
# 2. Create a copy of the WAHIS geographic lookup.
# ------------------------------------------------------------

geographic_lookup = wahis_admin[
    [
        "administrative_division",
        "admin_division_standard"
    ]
].copy()


# ------------------------------------------------------------
# 3. Create a geographic integration name.
#
# For known naming variations, use the validated boundary
# name.
#
# For all other records, retain the original WAHIS value.
# ------------------------------------------------------------

geographic_lookup["validated_boundary_name"] = (
    geographic_lookup["administrative_division"]
    .map(manual_geographic_mapping)
    .fillna(
        geographic_lookup["administrative_division"]
    )
)


# ------------------------------------------------------------
# 4. Classify the geographic record type.
#
# ADM1 records are suitable for spatial integration.
# Country-level records are not.
# ------------------------------------------------------------

geographic_lookup["geographic_level"] = np.where(
    geographic_lookup["administrative_division"]
    .str.strip()
    .str.lower()
    == "nigeria",
    "Country",
    "ADM1"
)


# ------------------------------------------------------------
# 5. Determine whether the record has a usable ADM1
#    boundary match.
# ------------------------------------------------------------

boundary_name_set = set(
    df_boundaries["boundary_name"]
    .dropna()
)


geographic_lookup["boundary_match"] = (
    geographic_lookup["validated_boundary_name"]
    .isin(boundary_name_set)
)


# ------------------------------------------------------------
# 6. Assign a final geographic validation status.
# ------------------------------------------------------------

geographic_lookup["geographic_match_status"] = np.select(
    [
        geographic_lookup["geographic_level"]
        == "Country",

        geographic_lookup["boundary_match"]
    ],
    [
        "Country-level — excluded from ADM1 spatial join",

        "Matched"
    ],
    default="Unmatched"
)


# ------------------------------------------------------------
# 7. Add a clear explanation for each status.
# ------------------------------------------------------------

geographic_lookup["validation_note"] = np.select(
    [
        geographic_lookup["administrative_division"]
        == "Nassarawa",

        geographic_lookup["administrative_division"]
        == "Federal Capital Territory",

        geographic_lookup["administrative_division"]
        == "Nigeria"
    ],
    [
        "Spelling variation mapped to Nasarawa",

        "Naming variation mapped to Abuja Federal Capital Territory",

        "Country-level record; no ADM1 boundary assigned"
    ],
    default="Standard ADM1 geographic match"
)


# ------------------------------------------------------------
# 8. Display the final validated lookup.
# ------------------------------------------------------------

display(
    geographic_lookup[
        [
            "administrative_division",
            "validated_boundary_name",
            "geographic_level",
            "boundary_match",
            "geographic_match_status",
            "validation_note"
        ]
    ]
)


# ------------------------------------------------------------
# 9. Calculate final validation statistics.
# ------------------------------------------------------------

total_geographic_values = len(
    geographic_lookup
)

matched_adm1 = (
    geographic_lookup[
        "geographic_match_status"
    ] == "Matched"
).sum()

country_level_records = (
    geographic_lookup[
        "geographic_level"
    ] == "Country"
).sum()

remaining_unmatched = (
    geographic_lookup[
        "geographic_match_status"
    ] == "Unmatched"
).sum()


print("=" * 60)
print("FINAL GEOGRAPHIC VALIDATION")
print("=" * 60)

print(
    f"Unique WAHIS geographic values: "
    f"{total_geographic_values:,}"
)

print(
    f"Validated ADM1 matches: "
    f"{matched_adm1:,}"
)

print(
    f"Country-level values: "
    f"{country_level_records:,}"
)

print(
    f"Remaining unmatched ADM1 values: "
    f"{remaining_unmatched:,}"
)


# ------------------------------------------------------------
# 10. Save the final geographic lookup.
# ------------------------------------------------------------

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "wahis_administrative_boundary_lookup.csv"
)

geographic_lookup.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"\nFinal geographic lookup saved to:\n{OUTPUT_PATH}"
)

,administrative_division,validated_boundary_name,geographic_level,boundary_match,geographic_match_status,validation_note
0,Abia,Abia,ADM1,True,Matched,Standard ADM1 geographic match
1,Adamawa,Adamawa,ADM1,True,Matched,Standard ADM1 geographic match
2,Akwa Ibom,Akwa Ibom,ADM1,True,Matched,Standard ADM1 geographic match
3,Anambra,Anambra,ADM1,True,Matched,Standard ADM1 geographic match
4,Bauchi,Bauchi,ADM1,True,Matched,Standard ADM1 geographic match
5,Bayelsa,Bayelsa,ADM1,True,Matched,Standard ADM1 geographic match
6,Benue,Benue,ADM1,True,Matched,Standard ADM1 geographic match
7,Borno,Borno,ADM1,True,Matched,Standard ADM1 geographic match
8,Cross River,Cross River,ADM1,True,Matched,Standard ADM1 geographic match
9,Delta,Delta,ADM1,True,Matched,Standard ADM1 geographic match


FINAL GEOGRAPHIC VALIDATION
Unique WAHIS geographic values: 39
Validated ADM1 matches: 38
Country-level values: 1
Remaining unmatched ADM1 values: 0

Final geographic lookup saved to:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_administrative_boundary_lookup.csv


In [9]:
# ============================================================
# NOTEBOOK 03 — FINAL GEOGRAPHIC VALIDATION
# ============================================================

# ------------------------------------------------------------
# 1. Validate that no genuine ADM1 divisions remain unmatched.
# ------------------------------------------------------------

remaining_unmatched_adm1 = geographic_lookup[
    (
        geographic_lookup["geographic_level"] == "ADM1"
    )
    &
    (
        geographic_lookup["geographic_match_status"]
        == "Unmatched"
    )
]

assert len(remaining_unmatched_adm1) == 0, (
    "Some ADM1 geographic divisions remain unmatched:\n"
    f"{remaining_unmatched_adm1}"
)


# ------------------------------------------------------------
# 2. Validate the known name harmonisations.
# ------------------------------------------------------------

nassarawa_mapping = geographic_lookup[
    geographic_lookup["administrative_division"]
    == "Nassarawa"
]

assert (
    nassarawa_mapping["validated_boundary_name"].iloc[0]
    == "Nasarawa"
), "Nassarawa was not correctly mapped to Nasarawa."


fct_mapping = geographic_lookup[
    geographic_lookup["administrative_division"]
    == "Federal Capital Territory"
]

assert (
    fct_mapping["validated_boundary_name"].iloc[0]
    == "Abuja Federal Capital Territory"
), (
    "Federal Capital Territory was not correctly "
    "mapped to Abuja Federal Capital Territory."
)


# ------------------------------------------------------------
# 3. Validate that Nigeria remains a country-level record.
# ------------------------------------------------------------

nigeria_mapping = geographic_lookup[
    geographic_lookup["administrative_division"]
    == "Nigeria"
]

assert (
    nigeria_mapping["geographic_level"].iloc[0]
    == "Country"
), (
    "Nigeria should be classified as a country-level "
    "geographic record."
)


assert (
    nigeria_mapping["boundary_match"].iloc[0]
    == False
), (
    "Nigeria must not be assigned to an ADM1 boundary."
)


# ------------------------------------------------------------
# 4. Calculate final statistics.
# ------------------------------------------------------------

adm1_records = geographic_lookup[
    geographic_lookup["geographic_level"] == "ADM1"
]

matched_adm1 = (
    adm1_records["boundary_match"].sum()
)

total_adm1 = len(adm1_records)

match_rate = (
    matched_adm1 / total_adm1 * 100
)


# ------------------------------------------------------------
# 5. Final report.
# ------------------------------------------------------------

print("=" * 60)
print("ADMINISTRATIVE BOUNDARIES VALIDATION COMPLETE")
print("=" * 60)

print(
    f"Total unique WAHIS geographic values: "
    f"{len(geographic_lookup):,}"
)

print(
    f"ADM1 geographic values: "
    f"{total_adm1:,}"
)

print(
    f"Validated ADM1 boundary matches: "
    f"{matched_adm1:,}"
)

print(
    f"Country-level values excluded from ADM1 join: "
    f"{len(geographic_lookup) - total_adm1:,}"
)

print(
    f"Final ADM1 matching rate: "
    f"{match_rate:.2f}%"
)

print(
    "\nAdministrative-boundaries validation completed successfully."
)

ADMINISTRATIVE BOUNDARIES VALIDATION COMPLETE
Total unique WAHIS geographic values: 39
ADM1 geographic values: 38
Validated ADM1 boundary matches: 38
Country-level values excluded from ADM1 join: 1
Final ADM1 matching rate: 100.00%

Administrative-boundaries validation completed successfully.
